In [1]:
from pathlib import Path
import pandas as pd
import sqlite3

In [2]:
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
file_path = project_root / "data" / "raw" / "online_retail_II.xlsx"

if not file_path.is_file():
    raise FileNotFoundError(f"원본 파일 위치를 확인하세요: {file_path}")

excel = pd.ExcelFile(file_path, engine="openpyxl")

print("파일: ", file_path)
print("시트 목록: ", excel.sheet_names)

파일:  /Users/seongjinseo/projects/ecommerce-customer-analysis/data/raw/online_retail_II.xlsx
시트 목록:  ['Year 2009-2010', 'Year 2010-2011']


In [3]:
db_path = project_root / "data" / "ecommerce.db"
conn = sqlite3.connect(db_path)

---

### 6. 상품별 판매 분석

상품별 판매 실적과 전체 판매액 대비 비중 집계 (상위 20개)

In [4]:
sql_path = project_root / "sql" / "04_product_sales.sql"

product_sales = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(product_sales)

,stock_code,quantity_sold,orders,sales_gbp,sales_share_pct
0,22423,26519,3918,331084.27,1.68
1,85123A,94323,5365,258065.91,1.31
2,85099B,96931,3989,180888.03,0.92
3,23843,80995,1,168469.60,0.86
4,47566,28216,2674,148395.98,0.75
5,84879,80293,2807,129681.08,0.66
6,22086,35126,2018,117884.19,0.60
7,23166,78033,247,81700.92,0.41
8,79321,15918,1135,80873.23,0.41
9,22197,88537,2322,79552.50,0.40


stock_code=23843은 주문 수가 1건인데 수량이 80,995로 4위를 차지하였다. 이유 분석

In [5]:
sql_path = project_root / "sql" / "05_check_product_transactions.sql"

check_product_transactions = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(check_product_transactions)

,invoice,invoice_date,description,customer_id,quantity,unit_price,line_amount
0,581483,2011-12-09 09:15:00,"PAPER CRAFT , LITTLE BIRDIE",16446,80995,2.08,168469.6
1,C581484,2011-12-09 09:27:00,"PAPER CRAFT , LITTLE BIRDIE",16446,-80995,2.08,-168469.6


In [ ]:
# sql_path = project_root / "sql" / "06_product_sales_with_negative.sql"

# product_sales_with_negative = pd.read_sql_query(
#     sql_path.read_text(encoding="utf-8"),
#     conn,
# )

# display(product_sales_with_negative)

,stock_code,positive_amount_gbp,negative_amount_gbp,signed_total_gbp
0,23843,168469.6,-168469.6,0.0


In [8]:
sql_path = project_root / "sql" / "06_product_sales_with_negative.sql"

product_sales_with_negative = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(product_sales_with_negative)

,stock_code,positive_amount_gbp,negative_amount_gbp,signed_total_gbp
0,22423,331084.27,-16570.80,314513.47
1,85123A,258065.91,-9387.10,248678.81
2,85099B,180888.03,-2168.82,178719.21
3,47566,148395.98,-1238.55,147157.43
4,84879,129681.08,-774.07,128907.01
5,22086,117884.19,-1461.70,116422.49
6,79321,80873.23,-635.75,80237.48
7,22197,79552.50,-616.58,78935.92
8,22386,76385.20,-866.98,75518.22
9,84347,71361.60,-330.45,71031.15


In [9]:
sql_path = project_root / "sql" / "07_product_negative_impact.sql"

product_negative_impact = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(product_negative_impact)

,stock_code,positive_amount_gbp,negative_amount_gbp,signed_total_gbp,negative_to_positive_pct
0,23843,168469.60,-168469.60,0.00,100.00
1,23166,81700.92,-77479.64,4221.28,94.83
2,22423,331084.27,-16570.80,314513.47,5.01
3,85123A,258065.91,-9387.10,248678.81,3.64
4,21843,65189.95,-5270.25,59919.70,8.08
5,85099B,180888.03,-2168.82,178719.21,1.20
6,22086,117884.19,-1461.70,116422.49,1.24
7,20725,71396.14,-1383.75,70012.39,1.94
8,47566,148395.98,-1238.55,147157.43,0.83
9,22386,76385.20,-866.98,75518.22,1.14


stock_code=23166은 양수 판매액 기준으로는 8위였지만, 음수 거래를 반영하면 금액이 크게 줄어든다. 확인 필요

In [10]:
sql_path = project_root / "sql" / "05b_check_product_23166.sql"

check_product_23166 = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(check_product_23166)

,invoice,invoice_date,description,customer_id,quantity,unit_price,line_amount
0,541431,2011-01-18 10:01:00,MEDIUM CERAMIC TOP STORAGE JAR,12346,74215,1.04,77183.60
1,C541433,2011-01-18 10:17:00,MEDIUM CERAMIC TOP STORAGE JAR,12346,-74215,1.04,-77183.60
2,561901,2011-07-31 15:42:00,MEDIUM CERAMIC TOP STORAGE JAR,14156,288,1.25,360.00
3,553607,2011-05-18 10:47:00,MEDIUM CERAMIC TOP STORAGE JAR,16684,240,1.04,249.60
4,C557508,2011-06-20 16:13:00,MEDIUM CERAMIC TOP STORAGE JAR,16684,-240,1.04,-249.60
5,561051,2011-07-24 13:11:00,MEDIUM CERAMIC TOP STORAGE JAR,16684,144,1.04,149.76
6,552882,2011-05-12 10:10:00,MEDIUM CERAMIC TOP STORAGE JAR,14646,96,1.04,99.84
7,554307,2011-05-23 14:59:00,MEDIUM CERAMIC TOP STORAGE JAR,12989,96,1.04,99.84
8,556917,2011-06-15 13:37:00,MEDIUM CERAMIC TOP STORAGE JAR,12415,96,1.04,99.84
9,563614,2011-08-18 08:51:00,MEDIUM CERAMIC TOP STORAGE JAR,12415,96,1.04,99.84


23166도 판매 직후 같은 금액의 취소가 발생한 사례가 존재한다. 원주문 연결 필드는 없지만, 해당 대형 판매가 전량 취소된 것으로 판단할 강한 근거가 된다.
이번 사례는 2011년 1월의 높은 객단가에도 영향을 주기 떄문에 5단계 결과의 보완 사항으로 기록한다.

In [11]:
sql_path = project_root / "sql" / "08_product_sales_concentration.sql"

product_sales_concentration = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(product_sales_concentration)

,top_n,top_sales_gbp,sales_share_pct
0,5,1086903.79,5.52
1,10,1576595.71,8.00
2,20,2256376.49,11.45


1. 상품별 판매액 순위는 판매 수량이나 주문 수 순위와 반드시 일치하지 않는다. 상품 단가와 주문당 구매 수량이 다르기 때문이다.
2. 또한 stock_code 23843과 23166에서는 대형 판매 직후 동일 금액의 취소가 확인되어 취소 차감 전 판매액만으로 상품 성과를 평가하면 과대 평가가 될 수 있다.
3. 상위 20개 상품이 전체 양수 상품 판매액의 11.45%를 차지했고, 나머지 상품에서 88.55%가 발생했다. 상위 20개 이외 상품도 전체 판매액에서 큰 비중을 차지한다. (단, 이는 취소 차감 전 기준이다.)

---

### 7. 고객별 구매 규모와 패턴을 파악하는 분석

고객별 구매 요약표를 만들고 양수 구매 금액 상위 20명을 조회

In [12]:
sql_path = project_root / "sql" / "09_customer_purchase_summary.sql"

customer_purchase_summary = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(customer_purchase_summary)

,customer_id,orders,quantity_purchased,purchase_amount_gbp,avg_order_value_gbp,first_purchase_date,last_purchase_date
0,18102,145,181645,580987.04,4006.81,2009-12-01 09:24:00,2011-12-09 11:50:00
1,14646,145,367072,526751.52,3632.77,2009-12-02 16:52:00,2011-12-08 12:12:00
2,14156,144,164400,303256.43,2105.95,2009-12-01 12:30:00,2011-11-30 10:54:00
3,14911,373,147842,272393.03,730.28,2009-12-01 11:41:00,2011-12-08 15:54:00
4,17450,51,83934,244944.25,4802.83,2010-09-27 16:59:00,2011-12-01 13:29:00
5,13694,143,188201,195640.69,1368.12,2009-12-04 15:26:00,2011-12-06 09:32:00
6,17511,60,117174,172132.87,2868.88,2009-12-02 10:52:00,2011-12-07 10:12:00
7,16446,2,80997,168472.50,84236.25,2011-05-18 09:52:00,2011-12-09 09:15:00
8,16684,55,104810,147142.77,2675.32,2009-12-07 12:56:00,2011-12-05 14:06:00
9,12415,24,91443,144033.37,6001.39,2010-06-30 08:30:00,2011-11-15 14:22:00


양수 구매 금액 상위 20명의 음수 거래 영향 비교

In [6]:
sql_path = project_root / "sql" / "10_customer_negative_impact.sql"

customer_negative_impact = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(customer_negative_impact)

,customer_id,positive_amount_gbp,negative_amount_gbp,signed_total_gbp,negative_to_positive_pct
0,16446,168472.50,-168469.60,2.90,100.00
1,12346,77352.96,-77183.60,169.36,99.78
2,12931,92347.34,-20800.37,71546.97,22.52
3,16029,109794.87,-14904.24,94890.63,13.57
4,14911,272393.03,-13791.56,258601.47,5.06
5,17450,244944.25,-11140.34,233803.91,4.55
6,17949,107833.48,-9364.42,98469.06,8.68
7,16684,147142.77,-5612.48,141530.29,3.81
8,14156,303256.43,-5294.87,297961.56,1.75
9,13694,195640.69,-5286.63,190354.06,2.70


1. customer_id가 16446, 12346은 음수 거래를 반영하면 금액이 각각 2.90, 169.36으로 줄어든다. 따라서 양수 구매 금액만으로 우수 고객을 선정하면 과대평가할 수 있다.
2. 18102와 14646은 구매 횟수가 많고, 음수 거래 반영 후에도 높은 구매 금액을 유지해 금액과 빈도 기준의 우수 고객 후보로 판단할 수 있다. 다만, 최종 분류에는 최근 구매 여부 등 추가 기준이 필요하다.

전체 고객의 구매 횟수 분포

In [32]:
sql_path = project_root / "sql" / "11_customer_order_distribution.sql"

customer_order_distribution = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(customer_order_distribution)

,order_count_group,customers,customer_share_pct
0,1,1618,27.65
1,2,945,16.15
2,3-5,1503,25.68
3,6-10,922,15.76
4,11+,864,14.76


1. 전체 구매 고객 5.852명 중 관측 기간 내 2회 이상 양수 상품 구매 기록이 있는 고객은 72.35%, 6회 이상은 30.52%였다. 반복 구매 기록이 있는 고객이 상당한 비중을 차지한다.
2. 하지만, 구매 후 나중에 취소된 원래의 양수 주문이 여전히 구매 횟수에 포함되어 있고(취소 영향), 2009년에 처음 관측된 고객은 약 2년 동안 구매할 기회가 있지만, 2011년 말에 처음 관측된 고객은 기간이 짧다. 즉, 구매 횟수를 같은 조건으로 비교한 결과가 아니다.(관측 기간 차이)

    이후 코호트 분석에서 구매 후 같은 기간을 기준으로 비교 예정 

양수 구매 금액 상위 5, 10, 20명의 금액이 전체 고객 식별 가능 구매 금액에서 차지하는 비중

In [36]:
sql_path = project_root / "sql" / "12_customer_purchase_concentration.sql"

customer_purchase_concentration = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(customer_purchase_concentration)

,top_n,top_purchase_amount_gbp,purchase_share_pct
0,5,1928332.27,11.26
1,10,2755754.47,16.09
2,20,3753065.90,21.92


전체 구매 고객(5,852명)의 약 0.34%인 상위 20명이 고객 식별 가능 양수 구매 금액의 21.92%를 차지해, 소수 고객의 금액 기여가 크게 나타났다. 
- 다만 일부 고객은 대형 취소의 영향으로 구매 금액이 과대평가되어 있어, 우수 고객 선정과 집중도 해석에는 음수 거래를 함께 고려해야 한다.

단, 
- 분모는 고객 ID가 있는 구매 금액이며, 고객 ID가 없는 판매는 제외된다.
- 상위 20명에는 대형 취소가 확인된 16446, 12346이 포함되어 있다. 취소 차감 전 집중도이다.

---

### 8. 월별 판매가 신규 고객과 기존 고객 중 어디에서 발생하는지 확인하는 분석

고객별 첫 구매 월을 구하고, 월별 신규/기존 구매 고객 수를 집계

In [44]:
sql_path = project_root / "sql" / "13_monthly_new_existing_customers.sql"

monthly_new_existing_customers = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(monthly_new_existing_customers)

,month,new_customers,existing_customers,total_customers,existing_customer_share_pct
0,2009-12,951,0,951,0.00
1,2010-01,368,333,701,47.50
2,2010-02,375,396,771,51.36
3,2010-03,441,610,1051,58.04
4,2010-04,294,645,939,68.69
5,2010-05,255,710,965,73.58
6,2010-06,267,767,1034,74.18
7,2010-07,185,739,924,79.98
8,2010-08,163,747,910,82.09
9,2010-09,239,897,1136,78.96


1. 2011년 11월 구매 고객 1,660명 중 기존 고객은 1,469명으로 88.49%였다. 즉, 그달 구매 고객 대부분은 이전에 구매한 뒤 다시 구매한 고객이다.
2. 2011년 10월 -> 11월 신규 고객은 30명 감소했지만, 기존 고객은 329명 증가해, 전체 구매 고객은 299명 증가했다. 전체 고객 수 증가는 기존 구매 고객 수 증가에서 나타났다.
- 다만 이 표만으로는 이전 달 구매자가 다음 달 구매자, 고객이 꾸준히 구매, 유지율이 높아졌다 등은 판단 불가

월별 신규/기존 고객의 구매 금액과 기여 비중을 비교

In [49]:
sql_path = project_root / "sql" / "14_monthly_new_existing_sales.sql"

monthly_new_existing_sales = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(monthly_new_existing_sales)

,month,new_sales_gbp,existing_sales_gbp,total_sales_gbp,existing_sales_share_pct
0,2009-12,681066.22,0.00,681066.22,0.00
1,2010-01,158694.20,376535.17,535229.37,70.35
2,2010-02,167352.32,332397.36,499749.69,66.51
3,2010-03,209982.61,458594.58,668577.19,68.59
4,2010-04,123664.93,464088.84,587753.77,78.96
5,2010-05,110056.82,484829.72,594886.54,81.50
6,2010-06,128733.47,502771.38,631504.85,79.61
7,2010-07,69497.34,513891.19,583388.53,88.09
8,2010-08,60995.80,535583.42,596579.22,89.78
9,2010-09,158953.12,649191.94,808145.06,80.33


1. 2011년 10월 -> 11월 전체 구매 금액 증가는 기존 고객의 구매 금액이 증가해서 증가했다.
2. 2011년 11월 기존 고객 수 비중(88.49%)보다 구매 금액 비중(92.95%)이 높아, 신규 고객보다 1명당 월평균 구매 금액이 높았다.

신규/기존 고객의 1인당 월 구매 금액을 주문 빈도와 객단가로 나눠 비교

In [52]:
sql_path = project_root / "sql" / "15_monthly_customer_group_metrics.sql"

monthly_customer_group_metrics = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(monthly_customer_group_metrics)

,month,customer_type,customers,orders,sales_gbp,orders_per_customer,avg_order_value_gbp,sales_per_customer_gbp
0,2009-12,new,951,1497,681066.22,1.57,454.95,716.16
1,2010-01,new,368,423,158694.20,1.15,375.16,431.23
2,2010-01,existing,333,536,376535.17,1.61,702.49,1130.74
3,2010-02,new,375,437,167352.32,1.17,382.96,446.27
4,2010-02,existing,396,656,332397.36,1.66,506.70,839.39
5,2010-03,new,441,525,209982.61,1.19,399.97,476.15
6,2010-03,existing,610,984,458594.58,1.61,466.05,751.79
7,2010-04,new,294,321,123664.93,1.09,385.25,420.63
8,2010-04,existing,645,994,464088.84,1.54,466.89,719.52
9,2010-05,new,255,285,110056.82,1.12,386.16,431.60


1. 2011년 11월 기준 고객은 신규 고객보다 고객당 주문 수와 객단가가 모두 높아, 1인당 월 구매 금액도 높았다.
2. 2011년 10월 -> 11월 전체 구매 금액은 증가했지만, 1인당 구매 금액은 줄었다. 이는 고객당 주문 수는 증가했지만 객단가가 하락해, 1인당 월 구매 금액은 795.61에서 722.72로 감소했다. 그러나 구매 고객 수가 1,140명에서 1,469명으로 늘어나면서 기존 고객의 전체 구매 금액은 증가했다. (즉, 한 명당 평균 금액은 줄었지만 구매한 사람이 더 많아져 전체 금액이 늘어난 것이다.)

### 9. 재구매 주기 / 코호트 분석

고객별 첫 주문과 두 번째 주문 사이의 기간

In [12]:
sql_path = project_root / "sql" / "16_first_to_second_purchase.sql"

first_to_second_purchase = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(first_to_second_purchase)

,customer_id,first_purchase_date,second_purchase_date,days_to_second_purchase
0,12346,2010-03-02 13:08:00,2010-06-28 13:53:00,118.03
1,12347,2010-10-31 14:20:00,2010-12-07 14:57:00,37.03
2,12348,2010-09-27 14:59:00,2010-12-16 19:09:00,80.17
3,12349,2010-04-29 13:20:00,2010-10-28 08:23:00,181.79
4,12350,2011-02-02 16:01:00,None,NaN
...,...,...,...,...
5847,18283,2010-02-19 17:16:00,2010-03-02 13:28:00,10.84
5848,18284,2010-10-04 11:33:00,None,NaN
5849,18285,2010-02-17 10:24:00,None,NaN
5850,18286,2009-12-16 10:45:00,2010-08-20 11:57:00,247.05


고객별로 상품 상세를 주문 단위로 묶고, 구매 일시순으로 첫 번째와 두 번째 주문을 구해 경과 일수를 계산했다.

전체 5,852명 중 두 번째 구매가 관측된 고객은 4,234명(72,35%), 한 번만 구매한 고객은 1,618명(27,65%)으로, 앞서 확인한 구매 횟수 분포와 일치했다.

첫 구매에서 두 번째 구매까지 걸린 기간의 분포

In [17]:
sql_path = project_root / "sql" / "17_second_purchase_interval_distribution.sql"

second_purchase_interval_distribution = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(second_purchase_interval_distribution)

,interval_group,customers,customer_share_pct
0,0-7일 미만,515,12.16
1,7-30일 미만,803,18.97
2,30-90일 미만,1358,32.07
3,90-180일 미만,829,19.58
4,180일 이상,729,17.22


- 재구매가 관측된 4,234명 중 30~90일 미만 구간이 1,358명(32,07%)으로 가장 많았다.
- 30일 미만에 두 번째 구매를 한 고객은 1,318명(31.13%)이다.

첫 구매 월별 코호트의 월간 구매 유지율

In [4]:
sql_path = project_root / "sql" / "18_monthly_cohort_retention.sql"

monthly_cohort_retention = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(monthly_cohort_retention)

,cohort_month,month_index,cohort_customers,active_customers,retention_pct
0,2009-12,0,951,951,100.00
1,2009-12,1,951,333,35.02
2,2009-12,2,951,317,33.33
3,2009-12,3,951,404,42.48
4,2009-12,4,951,360,37.85
...,...,...,...,...,...
295,2011-09,1,188,51,27.13
296,2011-09,2,188,71,37.77
297,2011-10,0,221,221,100.00
298,2011-10,1,221,70,31.67


1. 2009년 12월에 처음 구매한 것으로 관측된 고객 951면 중 333명(35.02%)이 2010년 1월에도 구매했다.
2. 같은 고객들이 매달 연속 구매했다고 단정할 수 없다. 3개월 후 구매 고객에는 중간 달에 구매하지 않았다가 돌아온 고객도 포함될 수 있다. 다만 모두 같은 코호트에 속하므로 첫 구매 이후의 재구매인 것은 맞다.

코호트별 1, 3, 6개월 후 구매 비율 비교

In [10]:
sql_path = project_root / "sql" / "19_cohort_retention_comparison.sql"

cohort_retention_comparison = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(cohort_retention_comparison)

,cohort_month,cohort_customers,month_1_retention_pct,month_3_retention_pct,month_6_retention_pct
0,2009-12,951,35.02,42.48,37.64
1,2010-01,368,21.47,31.52,26.90
2,2010-02,375,23.47,29.33,19.20
3,2010-03,441,19.05,24.26,24.72
4,2010-04,294,19.05,15.99,27.55
5,2010-05,255,15.69,17.65,21.18
6,2010-06,267,17.60,20.60,12.73
7,2010-07,185,15.68,29.73,11.35
8,2010-08,163,19.63,32.52,9.82
9,2010-09,239,23.01,12.97,13.81


1. 2011년 5월 코호트는 3개월 후 구매 비율이 16.22%였지만, 6개월 후에는 26.13%로 높아졌다. 같은 코호트에서 6개월 후 구매한 고객이 더 많았으며, 중간에 구매하지 않다가 돌아온 고객도 포함될 수 있다. 매달 연속 구매했다는 의미는 아니다.
2. 코호트 간 비교는 같은 경과 시점으로 맞춰야 한다. 1개월 후 구매 비율은 2011년 10월 코호트가 31.67%로, 5월 코호트의 23.42%보다 8.25%포인트 높았다. 다만 계절성과 고객 구성 차이가 있어, 이를 전반적인 고객 유지 성과 개선으로 단정할 수는 없다.

첫 구매 후 30일 이내 두 번째 구매 비율

In [21]:
sql_path = project_root / "sql" / "20_cohort_30day_repeat_rate.sql"

cohort_30day_repeat_rate = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(cohort_30day_repeat_rate)

,cohort_month,eligible_customers,repeat_within_30d_customers,repeat_within_30d_pct
0,2009-12,951,315,33.12
1,2010-01,368,74,20.11
2,2010-02,375,81,21.60
3,2010-03,441,77,17.46
4,2010-04,294,47,15.99
5,2010-05,255,45,17.65
6,2010-06,267,52,19.48
7,2010-07,185,37,20.00
8,2010-08,163,33,20.25
9,2010-09,239,52,21.76


1. 2011년 10월에 첫 구매한 것으로 관측된 고객 중 30일 관측 가능한 221명이 분모다. 이 중 61면(27.60%)이 첫 구매 후 30일 미만에 두 번쨰 구매를 했다.
2. 11월의 분석 대상 58명에서는 31.03%로, 10월보다 3.43%포인트 높았다. 다만 11월은 월초 첫 구매 고객 일부만 포함하고 고객 구성, 계절성도 다를 수 있어, 전반적인 재구매 성과가 개선됐다고 단정할 수는 없다.

### 10. RFM 고객 세분화

고객별 최근성, 구매 빈도, 구매 금액 요약

In [7]:
sql_path = project_root / "sql" / "21_customer_rfm_base.sql"

customer_rfm_base = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(customer_rfm_base)

,customer_id,last_purchase_date,recency_days,frequency,monetary_gbp
0,12346,2011-01-18 10:01:00,326,3,77352.96
1,12347,2011-12-07 15:52:00,3,8,4921.53
2,12348,2011-09-25 13:13:00,76,5,1658.40
3,12349,2011-11-21 09:51:00,19,3,3678.69
4,12350,2011-02-02 16:01:00,311,1,294.40
...,...,...,...,...,...
5847,18283,2011-12-06 12:02:00,4,22,2730.70
5848,18284,2010-10-04 11:33:00,432,1,411.68
5849,18285,2010-02-17 10:24:00,661,1,377.00
5850,18286,2010-08-20 11:57:00,477,2,1246.43


주요 고객 3명
- 12346: 326일, 3건, 77,352.96 파운드 (큰 구매 금액 대부분이 취소로 상쇄됨)
- 16446: 1일, 2건, 168,472.50 파운드 (최근 대형 구매가 취소돼 R과 M이 과대평가될 수 있음)
- 18102: 1일, 145건, 580,987.04 파운트 (최근 구매, 많은 주문, 높은 금액을 모두 기록함)

RFM 기본 지표에 음수 거래 영향을 연결하는 쿼리

In [19]:
sql_path = project_root / "sql" / "22_customer_rfm_negative_check.sql"

customer_rfm_negative_check = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(customer_rfm_negative_check)

,customer_id,recency_days,frequency,monetary_gbp,negative_amount_gbp,signed_total_gbp,negative_to_positive_pct
0,15935,240,1,108.04,-308.82,-200.78,285.84
1,13091,29,2,1108.80,-2452.04,-1343.24,221.14
2,13112,542,1,20.60,-26.04,-5.44,126.41
3,12467,424,1,132.80,-132.80,0.00,100.00
4,12558,8,1,269.96,-269.96,0.00,100.00
...,...,...,...,...,...,...,...
5847,18279,520,1,231.34,0.00,231.34,0.00
5848,18281,181,2,201.14,0.00,201.14,0.00
5849,18283,4,22,2730.70,0.00,2730.70,0.00
5850,18284,432,1,411.68,0.00,411.68,0.00


- 전체 5,852명 중 3,429명(58.60%)은 음수 거래 금액이 £0.00이었다. 
- 반면 고객 16446과 12346의 구매 금액은 음수 거래를 반영하면 각각 £2.90과 £169.36으로 줄었다. 
- 두 사례는 양수 구매 금액만으로 평가할 때 고객의 구매 규모가 과대평가될 수 있음을 보여준다.

고객별 R, F, M을 각각 1~5점으로 나누기

In [22]:
sql_path = project_root / "sql" / "23_customer_rfm_scores.sql"

customer_rfm_scores = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(customer_rfm_scores)

,customer_id,recency_days,frequency,monetary_gbp,r_score,f_score,m_score
0,12346,326,3,77352.96,2,3,5
1,12347,3,8,4921.53,5,4,5
2,12348,76,5,1658.40,3,4,4
3,12349,19,3,3678.69,5,3,5
4,12350,311,1,294.40,2,1,2
...,...,...,...,...,...,...,...
5847,18283,4,22,2730.70,5,5,4
5848,18284,432,1,411.68,1,2,2
5849,18285,661,1,377.00,1,2,2
5850,18286,477,2,1246.43,1,3,4


- 전체 고객을 최근성·주문 빈도·양수 구매 금액 기준으로 각각 5개 그룹으로 나눴다. 최근에 구매했을수록, 주문이 많을수록, 구매 금액이 클수록 높은 점수를 부여했다. 
- 고객 18102는 세 지표에서 모두 5점을 받았다. 이 점수를 기준으로 다음 쿼리에서 음수 거래 반영 전후의 M 점수 변화를 비교한다.

음수 거래를 반영했을 때 고객의 M 점수의 변화

In [24]:
sql_path = project_root / "sql" / "24_customer_m_score_comparison.sql"

customer_m_score_comparison = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(customer_m_score_comparison)

,customer_id,monetary_gbp,signed_total_gbp,m_score,adjusted_m_score,m_score_change
0,12346,77352.96,169.36,5,1,-4
1,12347,4921.53,4921.53,5,5,0
2,12348,1658.40,1658.40,4,4,0
3,12349,3678.69,3654.54,5,5,0
4,12350,294.40,294.40,2,2,0
...,...,...,...,...,...,...
5847,18283,2730.70,2730.70,4,4,0
5848,18284,411.68,411.68,2,2,0
5849,18285,377.00,377.00,2,2,0
5850,18286,1246.43,1138.43,4,3,-1


고객 16446과 12346은 양수 구매 금액 기준으로 모두 M 5점이었지만, 음수 거래 반영 후 총액이 각각 2.90과 169.36으로 줄어 두 고객 모두 1점으로 내려갔다.

18102는 음수 거래를 반영한 뒤에도 5점을 유지했다. 양수 구매 금액만으로는 점수를 매기면 대형 취소가 발생한 고객의 구매 규모를 과대평가할 수 있다.

고객별 RFM 점수를 고객군으로 분류

In [25]:
sql_path = project_root / "sql" / "25_customer_rfm_segments.sql"

customer_rfm_segments = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(customer_rfm_segments)

,customer_id,r_score,f_score,adjusted_m_score,rfm_segment
0,12346,2,3,1,나머지 고객
1,12347,5,4,5,핵심 고객 후보
2,12348,3,4,4,나머지 고객
3,12349,5,3,5,나머지 고객
4,12350,2,1,2,나머지 고객
...,...,...,...,...,...
5847,18283,5,5,4,핵심 고객 후보
5848,18284,1,2,2,나머지 고객
5849,18285,1,2,2,나머지 고객
5850,18286,1,3,3,나머지 고객


18102는 모두 5점이라 핵심 고객 후보로 분류된다.

고객군별 인원과 비중 집계

In [27]:
sql_path = project_root / "sql" / "26_customer_rfm_segment_summary.sql"

customer_rfm_segment_summary = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(customer_rfm_segment_summary)

,rfm_segment,customers,customer_share_pct
0,나머지 고객,3894,66.54
1,핵심 고객 후보,1281,21.89
2,최근 저빈도 고객,446,7.62
3,재활성화 후보,231,3.95


현재 RFM 기준으로는 전체 고객의 약 3분의 1을 핵심·재활성화·최근 저빈도 고객군으로 구분할 수 있었지만, 나머지 66.54%는 하나로 묶여 있어 이 안에서 어떤 고객을 우선 관리할지는 판단하기 어렵다.

고객군별 구매 특성 비교

In [29]:
sql_path = project_root / "sql" / "27_customer_rfm_segment_profile.sql"

customer_rfm_segment_profile = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(customer_rfm_segment_profile)

,rfm_segment,customers,avg_recency_days,avg_frequency,avg_signed_total_gbp
0,나머지 고객,3894,271.59,3.09,1010.27
1,재활성화 후보,231,341.24,9.05,3883.35
2,최근 저빈도 고객,446,28.21,1.47,537.22
3,핵심 고객 후보,1281,20.07,17.03,8848.34


- 재활성화 후보는 평균 9.05건의 주문 이력이 있지만 마지막 구매 후 평균 341일이 지났으므로, 재방문 안내를 보내고 반응을 확인할 수 있다.
- 최근 저빈도 고객은 마지막 구매 후 평균 28일이 지났고 누적 주문은 평균 1.47건이므로, 최근 구매 상품과 관련된 추천을 보내 추가 구매로 이어지는지 확인할 수 있다.

### 11. 데이터 처리 기준을 바꿔도 주요 결론이 유지되는지 확인 (결과 검증, 민감도 검증)

시트 내부에서 원본 값이 완전히 같은 행을 한 건으로 줄이면, 상품 판매액과 주문 수가 얼마나 달라지는가?

In [10]:
sql_path = project_root / "sql" / "28_duplicate_sensitivity.sql"

duplicate_sensitivity = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(duplicate_sensitivity)

,scenario,product_rows,orders,sales_gbp,avg_order_value_gbp
0,baseline,1015088,39516,19700954.46,498.56
1,deduplicated,1003357,39516,19643861.64,497.11


- 같은 시트에서 원본 값이 완전히 동일한 상품 상세 행을 한 건씩만 남기자, 주문 수는 39,516건으로 유지됐고 전체 양수 상품 판매액은 0.29%, 객단가는 1.45 감소했다.
- 전체 기간 집계에서 관측된 변화이며, 동일 행이 실제 중복 입력이었는지는 확인되지 않았다.

대형 취소 두 건이 해당 월의 판매액과 객단가에 얼마나 영향을 줬는지 확인

In [17]:
sql_path = project_root / "sql" / "29_large_cancellation_sensitivity.sql"

large_cancellation_sensitivity = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(large_cancellation_sensitivity)

,month,scenario,orders,sales_gbp,avg_order_value_gbp
0,2011-01,baseline,1081,671933.19,621.58
1,2011-12,baseline,816,615493.27,754.28
2,2011-01,adjusted,1080,594749.59,550.69
3,2011-12,adjusted,815,447023.67,548.50


- 앞서 '07'을 통해 확인된 대형 취소에 대응하는 양수 판매 상세를 제외하자, 2011년 1월 판매액의 전년 동월 대비 변화가(03) 9.46% 증가에서 3.11% 감소로 뒤집혔다.
- 2011년 12월에도 한 건을 제외하자 9일까지 집계된 판매액이 27.37% 줄고 객단가는 754.28에서 548.50으로 하락했다.
- 두 사례에서 대형 취소 한 건이 월별 판매액과 객단가 해석을 크게 바꿀 수 있음을 확인했다.

고객 ID가 없는 거래가 전체 양수 상품 판매액에서 차지하는 비중

In [20]:
sql_path = project_root / "sql" / "30_customer_id_coverage.sql"

customer_id_coverage = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(customer_id_coverage)

,customer_id_status,product_rows,row_share_pct,sales_gbp,sales_share_pct
0,identified,788206,77.65,17124941.00,86.92
1,missing,226882,22.35,2576013.46,13.08


- 고객 ID가 없는 거래는 전체 상품 상세 행의 22.35%, 양수 상품 판매액의 13.08%를 차지했다.
- 따라서 고객별 구매, 재구매, RFM 분석은 전체 양수 상품 판매액의 86.92%에 해당하는 식별 가능 거래를 바탕으로 한다. 
- 이 비율은 판매액 기준의 분석 범위이며, 전체 고객 중 몇 퍼센트를 식별했는지 알 수 없다.